# 09｜座標取整與歸站歧義
固定一般平日 08:00–08:59 電輔車。對已知起站旅次模擬兩種取整粒度，再用兩位網格對照缺 ID 旅次的候選站。候選站只來自本樣本，未驗證實際取整規則。

In [1]:
from zipfile import ZipFile
import pandas as pd

with ZipFile('../data/raw/202501-capitalbikeshare-tripdata.zip') as z:
    with z.open('202501-capitalbikeshare-tripdata.csv') as f:
        rides = pd.read_csv(f, usecols=['started_at','rideable_type','start_station_id','start_lat','start_lng'], dtype='string')
time = pd.to_datetime(rides.started_at, format='mixed', errors='coerce')
keep = (time.ge('2025-01-01') & time.lt('2025-02-01') & time.dt.hour.eq(8)
        & time.dt.dayofweek.lt(5) & ~time.dt.day.isin([1,9,20])
        & rides.rideable_type.eq('electric_bike'))
sample = rides.loc[keep].copy()
sample['station'] = sample.start_station_id.fillna('').str.strip()
coords = ['start_lat','start_lng']
sample[coords] = sample[coords].apply(pd.to_numeric, errors='coerce')
assert len(sample) == 11571
assert (sample.start_lat.between(-90,90) & sample.start_lng.between(-180,180)).all()
known = sample.loc[sample.station.ne('')]
unknown = sample.loc[sample.station.eq('')]
assert (len(known), len(unknown)) == (7100,4471)

def grid_summary(trips, decimals):
    return trips.assign(**trips[coords].round(decimals)).groupby(coords).station.agg(
        stations='nunique', trips='size')

rows = []
for decimals in (2,3):
    grids = grid_summary(known, decimals)
    multi = grids.stations.gt(1)
    assert grids.trips.sum() == len(known)
    rows.append([decimals,len(grids),int(multi.sum()),int(grids.loc[multi,'trips'].sum()),int(grids.stations.max())])
print(pd.DataFrame(rows, columns=['decimals','grids','multi_station_grids','known_trips_in_multi_grids','max_stations']).to_string(index=False))

# Compare published coarse coordinates against observed candidate stations.
candidates = grid_summary(known, 2).stations
matched = unknown.join(candidates.rename('candidates'), on=coords).candidates.fillna(0)
print('Unknown trips with zero / one / multiple observed candidates:',
      matched.eq(0).sum(), matched.eq(1).sum(), matched.gt(1).sum())


 decimals  grids  multi_station_grids  known_trips_in_multi_grids  max_stations
        2    204                  102                        6602            16
        3    573                    0                           0             1
Unknown trips with zero / one / multiple observed candidates: 396 760 3315
